# LG HelloDoctor — C팀 RAG 파이프라인 최종
> 크롤링 → 공공API → ChromaDB → Kakao · HIRA 병원검색 → 응급판단 → Tool Router

## Step 1 — 라이브러리 설치

In [9]:
!pip install chromadb sentence-transformers requests beautifulsoup4 python-dotenv -q
print('설치 완료!')

설치 완료!



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 2 — API 키 설정

In [10]:
from dotenv import load_dotenv
import os

load_dotenv()

KAKAO_API_KEY = os.getenv('KAKAO_API_KEY')
HIRA_API_KEY  = os.getenv('DATA_API_KEY')

DB_PATH = os.path.join(os.getcwd(), 'RAG', 'db')
os.makedirs(DB_PATH, exist_ok=True)
print(f'API 키 로드 완료!')
print(f'DB 경로: {DB_PATH}')

API 키 로드 완료!
DB 경로: c:\Users\juyeon\Desktop\project\LGHelloDoctor\RAG\db


## Step 3 — 국가건강정보포털 크롤링

In [11]:
!pip install selenium
!pip install webdriver-manager


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import requests
import re
import time

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
    'Referer': 'https://health.kdca.go.kr/healthinfo/biz/health/unifiedSearch/unifiedSearchMain.do'
}

# 새 URL 구조: POST gnrlzHealthInfoView.do (cntnts_sn은 2026년 4월 기준)
DISEASE_SNS = [
    ('무릎관절염',   '5540'),
    ('허리디스크',   '3348'),
    ('오십견',       '1567'),
    ('고혈압',       '6765'),
    ('당뇨병',       '5305'),
    ('심근경색',     '6770'),
    ('뇌졸중',       '5495'),
    ('위염',         '6777'),
    ('역류성식도염', '2057'),
    ('폐렴',         '5249'),
    ('천식',         '6784'),
    ('편두통',       '6557'),
    ('어지럼증',     '6550'),
    ('아토피',       '6582'),
    ('두드러기',     '6581'),
]

VIEW_URL = 'https://health.kdca.go.kr/healthinfo/biz/health/gnrlzHealthInfo/gnrlzHealthInfo/gnrlzHealthInfoView.do'

DISEASE_DEPT_MAP = {
    '무릎관절염': '정형외과', '허리디스크': '정형외과', '오십견': '정형외과',
    '고혈압': '내과',         '당뇨병': '내과',         '심근경색': '심장내과',
    '뇌졸중': '신경과',       '위염': '소화기내과',     '역류성식도염': '소화기내과',
    '폐렴': '내과',           '천식': '호흡기내과',     '편두통': '신경과',
    '어지럼증': '이비인후과', '아토피': '피부과',       '두드러기': '피부과',
}

def crawl_health_info(disease_name: str, cntnts_sn: str) -> dict:
    try:
        response = requests.post(
            VIEW_URL,
            headers=HEADERS,
            data={'cntnts_sn': cntnts_sn},
            timeout=10
        )
        response.encoding = 'utf-8'
        content_parts = []
        for div_id in ['contentsDiv3', 'contentsDiv4', 'contentsDiv5']:
            pattern = f'id="{div_id}"'
            if pattern in response.text:
                idx = response.text.index(pattern)
                section = response.text[idx:idx+3000]
                clean = re.sub(r'<[^>]+>', ' ', section)
                clean = re.sub(r'\s+', ' ', clean).strip()
                content_parts.append(clean[:400])
        content = ' '.join(content_parts)[:500].strip()
        return {'disease': disease_name, 'text': content, 'success': bool(content)}
    except Exception as e:
        return {'disease': disease_name, 'text': '', 'success': False}


crawled_docs = []
print('국가건강정보포털 크롤링 중...')
for name, sn in DISEASE_SNS:
    result = crawl_health_info(name, sn)
    time.sleep(0.5)
    if result['success']:
        dept = DISEASE_DEPT_MAP.get(name, '내과')
        crawled_docs.append({
            'id':       f'crawl_{name}',
            'text':     f"{name}: {result['text']} (진료과: {dept})",
            'category': '증상_진료과',
            'source':   'health.kdca.go.kr'
        })
        print(f'v {name}')
    else:
        print(f'x {name} 실패')

print(f'크롤링 완료: {len(crawled_docs)}개')


국가건강정보포털 크롤링 중...
v 무릎관절염
v 허리디스크
v 오십견
v 고혈압
v 당뇨병
v 심근경색
v 뇌졸중
v 위염
v 역류성식도염
v 폐렴
v 천식
v 편두통
v 어지럼증
v 아토피
v 두드러기
크롤링 완료: 15개


## Step 4 — 보완 문서 (크롤링 실패 대비 + 복약 · 응급)

In [13]:
MANUAL_DOCS = [
    {'id': 'manual_001', 'text': '무릎관절염은 정형외과에서 진료합니다. 관절 연골이 닳아 통증, 부종이 생깁니다.', 'category': '증상_진료과', 'source': 'manual'},
    {'id': 'manual_002', 'text': '고혈압은 내과에서 진료합니다. 혈압이 140/90mmHg 이상이면 고혈압입니다. 약을 임의로 끊으면 위험합니다.', 'category': '증상_진료과', 'source': 'manual'},
    {'id': 'manual_003', 'text': '당뇨병은 내과에서 진료합니다. 혈당 관리를 위해 식이요법, 운동, 약물치료를 병행합니다.', 'category': '증상_진료과', 'source': 'manual'},
    {'id': 'manual_004', 'text': '뇌졸중은 신경과에서 진료합니다. 갑자기 한쪽 팔다리 마비, 말이 어눌해지면 즉시 119에 신고하세요.', 'category': '응급_안내', 'source': 'manual'},
    {'id': 'manual_005', 'text': '심근경색은 심장내과에서 진료합니다. 가슴을 쥐어짜는 통증, 식은땀이 나면 즉시 119에 신고하세요.', 'category': '응급_안내', 'source': 'manual'},
    {'id': 'manual_006', 'text': '허리 통증은 정형외과 또는 신경외과에서 진료합니다. 디스크, 척추관협착증 등이 원인일 수 있습니다.', 'category': '증상_진료과', 'source': 'manual'},
    {'id': 'manual_007', 'text': '어깨 통증은 정형외과에서 진료합니다. 오십견, 회전근개 파열 등이 원인일 수 있습니다.', 'category': '증상_진료과', 'source': 'manual'},
    {'id': 'manual_008', 'text': '눈 통증, 시력 저하, 눈 충혈은 안과에서 진료합니다. 결막염, 녹내장, 백내장 등이 원인일 수 있습니다.', 'category': '증상_진료과', 'source': 'manual'},
    {'id': 'manual_009', 'text': '귀 통증, 이명, 난청은 이비인후과에서 진료합니다. 중이염, 이명증, 돌발성 난청 등이 원인일 수 있습니다.', 'category': '증상_진료과', 'source': 'manual'},
    {'id': 'manual_010', 'text': '피부 발진, 가려움증, 두드러기는 피부과에서 진료합니다. 아토피, 접촉성 피부염 등이 원인일 수 있습니다.', 'category': '증상_진료과', 'source': 'manual'},
    {'id': 'manual_011', 'text': '소변 통증, 혈뇨, 빈뇨는 비뇨의학과에서 진료합니다. 방광염, 요로결석 등이 원인일 수 있습니다.', 'category': '증상_진료과', 'source': 'manual'},
    {'id': 'manual_012', 'text': '치아 통증, 잇몸 출혈은 치과에서 진료합니다. 충치, 치주염 등이 원인일 수 있습니다.', 'category': '증상_진료과', 'source': 'manual'},
    {'id': 'manual_013', 'text': '혈압약은 매일 같은 시간에 복용해야 합니다. 임의로 중단하면 혈압이 급격히 상승할 수 있어 위험합니다.', 'category': '복약_안내', 'source': 'manual'},
    {'id': 'manual_014', 'text': '당뇨약은 종류에 따라 식전 또는 식후에 복용합니다. 저혈당 증상이 생기면 즉시 사탕이나 주스를 드세요.', 'category': '복약_안내', 'source': 'manual'},
    {'id': 'manual_015', 'text': '소염진통제는 위장 장애를 일으킬 수 있어 식후에 복용하세요. 혈압약과 함께 복용 시 주의가 필요합니다.', 'category': '복약_안내', 'source': 'manual'},
    {'id': 'manual_016', 'text': '항생제는 처방된 기간 동안 모두 복용해야 합니다. 임의로 중단하면 내성이 생길 수 있습니다.', 'category': '복약_안내', 'source': 'manual'},
]

# 크롤링된 항목과 중복 제거 후 합치기
crawled_names = [d['id'].replace('crawl_', '') for d in crawled_docs]
ALL_DOCUMENTS = crawled_docs + MANUAL_DOCS

print(f'전체 문서: {len(ALL_DOCUMENTS)}개')
print(f'  크롤링: {len(crawled_docs)}개')
print(f'  보완:   {len(MANUAL_DOCS)}개')

전체 문서: 31개
  크롤링: 15개
  보완:   16개


## Step 5 — ChromaDB 인덱스 구축

In [14]:
import chromadb
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer('jhgan/ko-sroberta-multitask')
print('임베딩 모델 로드 완료!')

chroma_client = chromadb.PersistentClient(path=DB_PATH)

try:
    chroma_client.delete_collection('medical_knowledge')
except:
    pass

collection = chroma_client.create_collection(
    name='medical_knowledge',
    metadata={'hnsw:space': 'cosine'}
)

texts      = [doc['text']     for doc in ALL_DOCUMENTS]
ids        = [doc['id']       for doc in ALL_DOCUMENTS]
metas      = [{'category': doc['category'], 'source': doc['source']} for doc in ALL_DOCUMENTS]
embeddings = embed_model.encode(texts).tolist()

collection.upsert(ids=ids, documents=texts, embeddings=embeddings, metadatas=metas)
print(f'ChromaDB 구축 완료: {collection.count()}개 저장')


c:\Users\juyeon\Desktop\project\LGHelloDoctor\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 33264.25it/s]
RobertaModel LOAD REPORT from: jhgan/ko-sroberta-multitask
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


임베딩 모델 로드 완료!
ChromaDB 구축 완료: 31개 저장


## Step 6 — RAG 파이프라인 (Query Rewriting + Hybrid Search + Reranking)

In [15]:
QUERY_REWRITE_MAP = {
    '무릎': '무릎관절염 정형외과 관절 통증 진료',
    '허리': '허리디스크 정형외과 척추 통증 진료',
    '어깨': '오십견 정형외과 어깨 통증 진료',
    '머리': '편두통 신경과 두통 진료',
    '배':   '위염 소화불량 소화기내과 진료',
    '가슴': '심근경색 심장내과 흉통 진료',
    '눈':   '안과 시력 결막염 진료',
    '귀':   '이비인후과 이명 난청 진료',
    '피부': '아토피 피부과 발진 진료',
    '소변': '비뇨의학과 방광 요로 진료',
    '혈압': '고혈압 내과 혈압약 복용',
    '당뇨': '당뇨병 내과 당뇨약 복용',
    '혈압약': '혈압약 복용 방법 주의사항',
    '당뇨약': '당뇨약 복용 방법 주의사항',
    '감기약': '감기약 복용 방법 주의사항',
}

def query_rewrite(query: str) -> str:
    for kw, rewritten in QUERY_REWRITE_MAP.items():
        if kw in query:
            return rewritten
    return query

def vector_search(query: str, n_results: int = 5) -> list:
    rewritten = query_rewrite(query)
    query_emb = embed_model.encode([rewritten]).tolist()
    results   = collection.query(query_embeddings=query_emb, n_results=n_results)
    return [{'text': results['documents'][0][i], 'category': results['metadatas'][0][i]['category'], 'source': results['metadatas'][0][i]['source']} for i in range(len(results['documents'][0]))]

def keyword_search(query: str) -> list:
    keywords = query.split()
    matched  = []
    for doc in ALL_DOCUMENTS:
        score = sum(1 for kw in keywords if kw in doc['text'])
        if score > 0:
            matched.append({'text': doc['text'], 'category': doc['category'], 'source': doc['source'], 'score': score})
    matched.sort(key=lambda x: x['score'], reverse=True)
    return matched[:3]

def rerank(query: str, docs: list) -> list:
    from numpy import dot
    from numpy.linalg import norm
    q_emb    = embed_model.encode([query])
    d_embs   = embed_model.encode([d['text'] for d in docs])
    scores   = [(dot(q_emb[0], d_emb) / (norm(q_emb[0]) * norm(d_emb)), docs[i]) for i, d_emb in enumerate(d_embs)]
    scores.sort(key=lambda x: x[0], reverse=True)
    return [doc for _, doc in scores]

def full_rag_pipeline(query: str) -> str:
    vector_results  = vector_search(query, n_results=5)
    keyword_results = keyword_search(query)
    seen = set()
    combined = []
    for r in vector_results + keyword_results:
        if r['text'] not in seen:
            seen.add(r['text'])
            combined.append(r)
    reranked = rerank(query, combined)
    return ' '.join([r['text'] for r in reranked[:3]])

print('RAG 파이프라인 준비 완료!')

RAG 파이프라인 준비 완료!


## Step 7 — 병원 검색 (Kakao + HIRA 통합)

In [16]:
SYMPTOM_DEPT_MAP = {
    '무릎': ('정형외과', '05'), '허리': ('정형외과', '05'),
    '어깨': ('정형외과', '05'), '발목': ('정형외과', '05'),
    '눈':   ('안과', '12'),     '귀':   ('이비인후과', '13'),
    '코':   ('이비인후과', '13'), '목':  ('이비인후과', '13'),
    '치아': ('치과', '21'),     '잇몸': ('치과', '21'),
    '피부': ('피부과', '14'),   '소변': ('비뇨의학과', '15'),
    '머리': ('신경과', '02'),   '가슴': ('심장내과', '01'),
    '배':   ('소화기내과', '01'), '당뇨': ('내과', '01'),
    '혈압': ('내과', '01'),     '기침': ('내과', '01'),
}

def search_kakao(dept_name: str, lat: float = 37.5012, lng: float = 127.0396) -> list:
    url     = 'https://dapi.kakao.com/v2/local/search/keyword.json'
    headers = {'Authorization': f'KakaoAK {KAKAO_API_KEY}'}
    params  = {'query': dept_name, 'x': lng, 'y': lat, 'radius': 2000, 'category_group_code': 'HP8', 'size': 5}
    try:
        response = requests.get(url, headers=headers, params=params, timeout=5)
        return [{'name': p['place_name'], 'address': p['road_address_name'] or p['address_name'], 'phone': p['phone'], 'distance': int(p['distance']), 'source': 'kakao'} for p in response.json().get('documents', [])]
    except:
        return []

def search_hira(dept_code: str, sido: str = '110000') -> list:
    url    = 'https://apis.data.go.kr/B551182/hospInfoServicev2/getHospBasisList'
    params = {'serviceKey': HIRA_API_KEY, 'pageNo': 1, 'numOfRows': 5, 'sidoCd': sido, 'dgsbjtCd': dept_code}
    try:
        response = requests.get(url, params=params, timeout=10)
        root     = ET.fromstring(response.text)
        return [{'name': item.findtext('yadmNm', ''), 'address': item.findtext('addr', ''), 'phone': item.findtext('telno', ''), 'source': 'hira'} for item in root.findall('.//item')]
    except:
        return []

def search_hospital(symptom_text: str, lat: float = 37.5012, lng: float = 127.0396) -> dict:
    dept_name, dept_code = '내과', '01'
    for symptom, (name, code) in SYMPTOM_DEPT_MAP.items():
        if symptom in symptom_text:
            dept_name, dept_code = name, code
            break
    kakao = sorted([h for h in search_kakao(dept_name, lat, lng) if h['phone']], key=lambda x: x['distance'])
    hira  = search_hira(dept_code)
    return {'department': dept_name, 'nearby': kakao[:3], 'official': hira[:3]}

print('병원 검색 준비 완료!')

병원 검색 준비 완료!


## Step 8 — 응급 판단 (심각도 점수화)

In [17]:
EMERGENCY_SCORES = {
    '숨이 안 쉬어': 100, '의식이 없': 100, '심장이 멎': 100,
    '피를 토': 90,       '가슴이 너무 아프': 90, '한쪽이 마비': 90,
    '말이 어눌': 85,     '입이 돌아': 85,
    '갑자기 안 보여': 80, '쓰러': 80, '혈압이 200': 80,
    '약을 잘못': 75,     '뼈가 부러': 70, '화상': 65,
    '식은땀': 30,        '가슴이 아파': 40,
    '어지러': 20,        '두통': 15,
}

def emergency_check(text: str) -> dict:
    total, matched = 0, []
    for kw, score in EMERGENCY_SCORES.items():
        if kw in text:
            total += score
            matched.append(kw)
    if len(matched) >= 2:
        total = min(total * 1.2, 100)
    if total >= 70:
        return {'is_emergency': True,  'severity': 'HIGH',   'score': round(total), 'action': '지금 바로 119에 전화해 주세요.'}
    elif total >= 40:
        return {'is_emergency': True,  'severity': 'MEDIUM', 'score': round(total), 'action': '응급실에 가보시는 게 좋을 것 같아요.'}
    else:
        return {'is_emergency': False, 'severity': 'LOW',    'score': round(total), 'action': None}

print('응급 판단 준비 완료!')

응급 판단 준비 완료!


## Step 9 — Tool Router

In [18]:
def tool_router(input_from_B: dict, lat: float = 37.5012, lng: float = 127.0396) -> dict:
    """
    B팀 출력 받아서 도구 선택 후 실행
    → D팀에 전달
    """
    intent = input_from_B.get('intent', 'symptom_inquiry')
    query  = input_from_B.get('query', '')

    result = {'intent': intent, 'rag_context': None, 'hospitals': None, 'emergency': None}

    # 응급 먼저 체크
    emerg = emergency_check(query)
    if emerg['is_emergency']:
        result['emergency'] = emerg
        if emerg['severity'] == 'HIGH':
            return result

    if intent == 'symptom_inquiry':
        result['rag_context'] = full_rag_pipeline(query)
        result['hospitals']   = search_hospital(query, lat, lng)
    elif intent == 'medication_info':
        result['rag_context'] = full_rag_pipeline(query)
    elif intent == 'hospital_search':
        result['hospitals'] = search_hospital(query, lat, lng)

    return result

print('Tool Router 준비 완료!')

Tool Router 준비 완료!


## Step 10 — 최종 통합 테스트

In [19]:
scenarios = [
    {'name': '시나리오 A: 무릎 통증', 'input': {'intent': 'symptom_inquiry', 'query': '무릎이 너무 아파요'}},
    {'name': '시나리오 B: 응급',      'input': {'intent': 'emergency',        'query': '가슴이 아프고 숨이 안 쉬어져요'}},
    {'name': '시나리오 C: 복약',      'input': {'intent': 'medication_info',  'query': '혈압약이랑 감기약 같이 먹어도 되나요'}},
    {'name': '시나리오 D: 병원 검색', 'input': {'intent': 'hospital_search',  'query': '가까운 정형외과 알려주세요'}},
]

for s in scenarios:
    print('=' * 50)
    print(s['name'])
    print('=' * 50)
    result = tool_router(s['input'])

    if result['rag_context']:
        print(f'RAG: {result["rag_context"][:80]}...')
    if result['hospitals']:
        print(f'진료과: {result["hospitals"]["department"]}')
        print(f'가까운 병원: {len(result["hospitals"]["nearby"])}개')
    if result['emergency']:
        print(f'응급: {result["emergency"]["severity"]} ({result["emergency"]["score"]}점)')
        print(f'행동: {result["emergency"]["action"]}')
    print()

시나리오 A: 무릎 통증
RAG: 무릎관절염은 정형외과에서 진료합니다. 관절 연골이 닳아 통증, 부종이 생깁니다. 무릎관절염: id="contentsDiv3" class="con...
진료과: 정형외과
가까운 병원: 3개

시나리오 B: 응급
응급: HIGH (100점)
행동: 지금 바로 119에 전화해 주세요.

시나리오 C: 복약
RAG: 소염진통제는 위장 장애를 일으킬 수 있어 식후에 복용하세요. 혈압약과 함께 복용 시 주의가 필요합니다. 고혈압은 내과에서 진료합니다. 혈압이 1...

시나리오 D: 병원 검색
진료과: 내과
가까운 병원: 3개

